# VisiPruner · Middle Mask / Preserve trên toàn bộ test

Chạy 8 cấu hình trên 16.817 câu DriveLM test, greedy, tối đa 512 token mới. Các can thiệp chỉ áp dụng ở decoder layer 20–24. Dùng lại điểm baseline toàn tập từ Deep Discard.

Bật GPU và Internet. T4 ×2 chạy song song hai phiên bản của cùng một mask. Thứ hạng token được lấy từ prefill gốc và dùng chung cho các lượt. Dự đoán được lưu sau mỗi bộ ảnh.

In [ ]:
%%writefile /kaggle/working/visipruner-middle-full.env
REPOSITORY=https://github.com/huyvanzzz/Log_Metrics.git
SOURCE_BRANCH=smolvlm-attention-nguyet
SOURCE_DIR=/kaggle/working/log-metrics-visipruner
DATA_DIR=/kaggle/working/drivelm-nuscenes-v1-0
export HF_TOKEN_SECRET=''
OUTPUT_DIR=/kaggle/working/visipruner_middle_20_24_preserve_512_outputs
RESUME_FROM=''
BATCH_SIZE=8
MAX_NEW_TOKENS=512
BASELINE_SCORES=/kaggle/working/scores-deep-discard-baseline.csv


## Cài đặt

Cập nhật mã nguồn và cài thư viện chạy mô hình, chấm điểm và vẽ hình.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-middle-full.env
if [ -d "$SOURCE_DIR/.git" ]; then
  test "$(git -C "$SOURCE_DIR" branch --show-current)" = "$SOURCE_BRANCH"
  test -z "$(git -C "$SOURCE_DIR" status --porcelain --untracked-files=no)"
  git -C "$SOURCE_DIR" pull --ff-only origin "$SOURCE_BRANCH"
else
  git clone --depth 1 --single-branch --branch "$SOURCE_BRANCH" "$REPOSITORY" "$SOURCE_DIR"
fi
python -m pip install -q -r "$SOURCE_DIR/smolvlm_attention/requirements.txt" pycocoevalcap==1.2
if ! command -v java >/dev/null; then
  apt-get update -qq
  apt-get install -y -qq default-jre-headless
fi


## Khôi phục phiên trước (tùy chọn)

Nếu chia qua nhiều phiên, lưu thư mục đầu ra, gắn bản lưu làm Kaggle Input và đặt `RESUME_FROM` thành thư mục chứa `manifest.json`. Cell dưới sao chép vào thư mục đầu ra mới. Với đầu ra đã có trong phiên hiện tại, chạy lại các cell đo để tiếp tục từ bộ ảnh cuối đã lưu.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-middle-full.env
cd "$SOURCE_DIR"
if [ -n "$RESUME_FROM" ] && [ ! -e "$OUTPUT_DIR" ]; then
  test -f "$RESUME_FROM/manifest.json"
  mkdir -p "$(dirname "$OUTPUT_DIR")"
  staging=$(mktemp -d "${OUTPUT_DIR}.restore.XXXXXX")
  trap 'rm -rf -- "$staging"' EXIT
  cp -R -- "$RESUME_FROM"/. "$staging/"
  chmod -R u+rwX -- "$staging"
  mv -T -- "$staging" "$OUTPUT_DIR"
  trap - EXIT
fi


## Điểm baseline dùng lại

Hàng baseline lấy từ `docs/ablation-visipruner/results/scores-deep-discard-scale.txt` (định dạng CSV). Có thể đặt `BASELINE_SCORES` thành đường dẫn file score Deep Discard đã gắn trên Kaggle.

In [ ]:
%%writefile /kaggle/working/scores-deep-discard-baseline.csv
variant,bleu4,meteor,rouge_l,cider,overall,yes_no_accuracy,removed_visual_mass_share,fallback_share,truncated,delta_overall,delta_yes_no
baseline,52.18577794659268,38.6315558392186,75.51595287086458,341.1000826891379,50.11082373139741,85.70258090946334,,,99,0.0,0.0


## Chuẩn bị dữ liệu

Dùng toàn bộ split test, giữ nguyên thứ tự và ID câu hỏi.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-middle-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.mvpruner_ablation fetch-hf --output "$DATA_DIR"
python -u -m smolvlm_attention.visipruner_ablation prepare --suite middle-full --test "$DATA_DIR/multi_frame/multi_frame_test.json" --image-root "$DATA_DIR" --output "$OUTPUT_DIR"


## 01–02/08 · Layer 20–24 · Chỉ giữ top 5%

Chạy cùng tập token được giữ hoặc mask, lần lượt với trọng số gốc và với tổng attention lên ảnh được bảo toàn.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-middle-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants middle-20-24-keep5 middle-20-24-keep5-mass --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 03–04/08 · Layer 20–24 · Chỉ giữ top 10%

Chạy cùng tập token được giữ hoặc mask, lần lượt với trọng số gốc và với tổng attention lên ảnh được bảo toàn.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-middle-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants middle-20-24-keep10 middle-20-24-keep10-mass --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 05–06/08 · Layer 20–24 · Mask top 10%

Chạy cùng tập token được giữ hoặc mask, lần lượt với trọng số gốc và với tổng attention lên ảnh được bảo toàn.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-middle-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants middle-20-24-top10 middle-20-24-top10-mass --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## 07–08/08 · Layer 20–24 · Mask top 5%

Chạy cùng tập token được giữ hoặc mask, lần lượt với trọng số gốc và với tổng attention lên ảnh được bảo toàn.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-middle-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants middle-20-24-top5 middle-20-24-top5-mass --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## Chấm điểm và vẽ hình

Tính delta so với baseline đã nhập. Xuất hai hình PNG/PDF cho Overall và Yes/No accuracy, đặt hai chế độ mask cạnh nhau.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-middle-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation score --output "$OUTPUT_DIR" --baseline-scores "$BASELINE_SCORES"


In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-middle-full.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation plot --output "$OUTPUT_DIR"
